## Setup: install GeoPandas (once)

This is the same as `%pip install pandas` in Lab 1. It can take a minute or two. If the next cell then says it cannot find geopandas, restart the kernel (the Restart button at the top of the notebook) and run it again. If the install itself fails, show your TA the last red line.

In [58]:
%pip install geopandas

62109.79s - pydevd: Sending message related to process being replaced timed-out after 5 seconds



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import geopandas as gpd
import json, os
print("GeoPandas", gpd.__version__, "is ready")

GeoPandas 1.1.4 is ready


*Importing the actual data*
 to get an idea of how to go about it moving foward 

In [ ]:
gdf = gpd.read_file("../Data/Aire_repartition_populations_caribouForestier.geojson")

Figuring out the size and main content of the file because she huge bro 

In [ ]:
print("Total habitat features:", len(gdf))
print(gdf.geom_type.value_counts())
print(gdf["NOM_ANGLA"].dropna().unique()[:20])

What organization and cordinate system is used in the data 

In [ ]:
print("CRS:", gdf.crs)
print("Columns:", gdf.columns.tolist())

### Recipe 3. A shapefile, GeoPackage or GeoJSON (points, lines or areas)

A shapefile is several files with the same name (`.shp`, `.shx`, `.dbf`, `.prj` and more); keep them together in `data/` and point at the `.shp`. GeoPandas reads the coordinate system from the `.prj`. Lines and areas are often far more detailed than a web map needs: `simplify` removes points closer together than the tolerance, **in the file's own units**, so do it before converting to degrees.

In [ ]:
gdf = gpd.read_file("../Data/Aire_repartition_populations_caribouForestier.geojson")      # or .gpkg, or .geojson
print(gdf.crs, len(gdf), sorted(gdf.geom_type.unique()))
print(gdf.columns.tolist())

gdf= gdf.rename(
    columns={"LEMV": "status", "NOM_POP": "Local_Pop_Name", "NOM_FRANCA": "name_fr", "NOM_ANGLA": "name_en", "NOM_SCIEN": "name_sci", "Rang_S": "Level_Of_Stability", "Tempo": "Years_of_Study", "Sup_KM2": "area_km2", "Sup_HA": "area_ha", "DATE_MAJ": "date_updated"})

gdf = gdf[["Local_Pop_Name", "name_fr", "name_en", "name_sci", "Level_Of_Stability", "status", "Years_of_Study", "area_km2", "area_ha", "date_updated", "geometry"]]# # Lines or areas only: simplify to about 5 metres if the file is in metres (skip if gdf.crs says degrees)

gdf["geometry"] = gdf.geometry.simplify(5)
gdf = gdf.to_crs("EPSG:4326")
gdf.to_file("../Data/Caribou_Simplified.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)


In [53]:
gdf.shape

NameError: name 'gdf' is not defined

### The check cell you will reuse for your own data

Change only the four settings at the top when you use it on your project. It reads the file back the way the browser will.

In [54]:
# ---- settings: change these for your own data ----
FILE = "../Data/Cleaned/Caribou_Simplified.geojson"
WEST, EAST, SOUTH, NORTH =  -82.64, 47.15, -58.94, 55.11  # a box around your area, in degrees
EXPECTED = 15          # how many features you expect (rows after your filters)
POPUP_FIELDS = ["Local_Pop_Name", "name_fr", "name_en", "name_sci", "Level_Of_Stability", "status", "area_km2", "area_ha"]  # which fields you want to show in the popup
# ----------------------------------------------------

back = gpd.read_file(FILE)
problems = []
if len(back) != EXPECTED:
    problems.append(f"{len(back)} features, but you expected {EXPECTED}.")
if back.crs is None or back.crs.to_epsg() != 4326:
    problems.append(f"The file is in {back.crs}, not longitude and latitude (EPSG:4326).")
minx, miny, maxx, maxy = back.total_bounds
if not (WEST <= minx and maxx <= EAST and SOUTH <= miny and maxy <= NORTH):
    problems.append(f"Some features are outside your box: x {minx:.3f} to {maxx:.3f}, y {miny:.3f} to {maxy:.3f}.")
missing = [f for f in POPUP_FIELDS if f not in back.columns]
if missing:
    problems.append(f"Missing popup fields: {missing}.")
mb = os.path.getsize(FILE) / 1_000_000
if mb > 5:
    problems.append(f"The file is {mb:.1f} MB; over about 5 MB the page gets slow. Filter or simplify.")
print("Geometry types:", sorted(back.geom_type.unique()))
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

Geometry types: ['Polygon']
All checks passed


Importing my second base data 

In [ ]:
gdf_gaspesie = gpd.read_file("../Data/Aire_repartition_caribouMontagnard_Gaspesie.geojson")

Getting the basic infomation on the data

In [ ]:
print("Total habitat features:", len(gdf_gaspesie))
print(gdf_gaspesie.geom_type.value_counts())
print(gdf_gaspesie["NOM_ANGLA"].dropna().unique()[:20])
print("CRS:", gdf_gaspesie.crs)
print("Columns:", gdf_gaspesie.columns.tolist())

### Recipe 3 with data 2. A shapefile, GeoPackage or GeoJSON (points, lines or areas)

A shapefile is several files with the same name (`.shp`, `.shx`, `.dbf`, `.prj` and more); keep them together in `data/` and point at the `.shp`. GeoPandas reads the coordinate system from the `.prj`. Lines and areas are often far more detailed than a web map needs: `simplify` removes points closer together than the tolerance, **in the file's own units**, so do it before converting to degrees.

In [ ]:
gdf_gaspesie = gpd.read_file("../Data/Aire_repartition_caribouMontagnard_Gaspesie.geojson")      # or .gpkg, or .geojson
print(gdf_gaspesie.crs, len(gdf_gaspesie), sorted(gdf_gaspesie.geom_type.unique()))
print(gdf_gaspesie.columns.tolist())

gdf_gaspesie = gdf_gaspesie.rename(
    columns={"LEMV": "status", "NOM_POP": "Local_Pop_Name", "NOM_FRANCA": "name_fr", "NOM_ANGLA": "name_en", "NOM_SCIEN": "name_sci", "Rang_S": "Level_Of_Stability", "Tempo": "Years_of_Study", "SUP_KM2": "area_km2", "SUP_HA": "area_ha", "DATE_MAJ": "date_updated"})

gdf_gaspesie = gdf_gaspesie[["Local_Pop_Name", "name_fr", "name_en", "name_sci", "Level_Of_Stability", "status", "Years_of_Study", "area_km2", "area_ha", "date_updated", "geometry"]]# # Lines or areas only: simplify to about 5 metres if the file is in metres (skip if gdf.crs says degrees)

gdf_gaspesie["geometry"] = gdf_gaspesie.geometry.simplify(5)
gdf_gaspesie = gdf_gaspesie.to_crs("EPSG:4326")
gdf_gaspesie.to_file("../Data/Caribou_Gaspesie_Simplified.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)


In [ ]:
gdf_gaspesie.shape

### The check cell you will reuse for your own data

Change only the four settings at the top when you use it on your project. It reads the file back the way the browser will.

In [55]:
# ---- settings: change these for your own data ----
FILE = "../Data/Cleaned/Caribou_Gaspesie_Simplified.geojson"
WEST, EAST, SOUTH, NORTH =  -82.64, 47.15, -58.94, 55.11  # a box around your area, in degrees
EXPECTED = 1          # how many features you expect (rows after your filters)
POPUP_FIELDS = ["Local_Pop_Name", "name_fr", "name_en", "name_sci", "Level_Of_Stability", "status", "area_km2", "area_ha",]  # which fields you want to show in the popup
# ----------------------------------------------------

back = gpd.read_file(FILE)
problems = []
if len(back) != EXPECTED:
    problems.append(f"{len(back)} features, but you expected {EXPECTED}.")
if back.crs is None or back.crs.to_epsg() != 4326:
    problems.append(f"The file is in {back.crs}, not longitude and latitude (EPSG:4326).")
minx, miny, maxx, maxy = back.total_bounds
if not (WEST <= minx and maxx <= EAST and SOUTH <= miny and maxy <= NORTH):
    problems.append(f"Some features are outside your box: x {minx:.3f} to {maxx:.3f}, y {miny:.3f} to {maxy:.3f}.")
missing = [f for f in POPUP_FIELDS if f not in back.columns]
if missing:
    problems.append(f"Missing popup fields: {missing}.")
mb = os.path.getsize(FILE) / 1_000_000
if mb > 5:
    problems.append(f"The file is {mb:.1f} MB; over about 5 MB the page gets slow. Filter or simplify.")
print("Geometry types:", sorted(back.geom_type.unique()))
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

Geometry types: ['Polygon']
All checks passed


Working again but with the other data 
The disturbances dataset is available by species only this means I have 6 diffent folders with the data so for this particular data set I will be going through them to seee if they allign and if i can merge them unto one layer.


Shows me what is in all of the folders

In [ ]:

from pathlib import Path
gdb_paths = sorted(Path("../Data").glob("*.gdb"))
reports = []
for gdb_path in gdb_paths:
        layers = gpd.list_layers(str(gdb_path))
        for layer_name in layers["name"]:
              layer_data = gpd.read_file(
                str(gdb_path),
                layer=layer_name
            )
              reports.append({
    "file": gdb_path.name,
    "layer": layer_name,
    "features": len(layer_data),
    "crs": str(layer_data.crs),
    "geometry_types": ", ".join(sorted(layer_data.geom_type.dropna().unique())),
    "columns": ", ".join(layer_data.columns)
})
comparison = pd.DataFrame(reports)
display(comparison)
        

Function to Exports the disturbance layer and created a geojson

In [ ]:
from pathlib import Path

def export_disturbance_layer(gdb_path, output_folder, tolerance=25):
    gdb_path = Path(gdb_path)
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    layer_info = gpd.list_layers(str(gdb_path))

    matching_layers = [
        name for name in layer_info["name"]
        if name.endswith("_Disturb_Perturb_Poly")
    ]

    if len(matching_layers) != 1:
        raise ValueError(
            f"Expected one disturbance layer in {gdb_path.name}, "
            f"but found: {matching_layers}"
        )

    layer_data = gpd.read_file(
        str(gdb_path),
        layer=matching_layers[0]
    )

    layer_data = layer_data.to_crs("EPSG:32198")

    layer_data["geometry"] = layer_data.geometry.simplify(
        tolerance,
        preserve_topology=True
    )

    layer_data = layer_data.to_crs("EPSG:4326")

    columns_to_keep = [
        column for column in ["Class", "geometry"]
        if column in layer_data.columns
    ]

    layer_data = layer_data[columns_to_keep].copy()

    output_file = output_folder / (
        gdb_path.stem + "_Disturb_Perturb_Poly.geojson"
    )

    layer_data.to_file(
        output_file,
        driver="GeoJSON",
        COORDINATE_PRECISION=5
    )

    print("Created:", output_file)
    print("Features:", len(layer_data))

    return layer_data

using the function 

In [ ]:
charlevoix_disturbance = export_disturbance_layer(
    "../Data/Charlevoix2020.gdb",
    "../Data/Take2/",
    tolerance=25
)

In [7]:
# ---- settings: change these for your own data ----
FILE = "../Data/Take2/Charlevoix2020_Disturb_Perturb_Poly.geojson"
WEST, EAST, SOUTH, NORTH =  -82.64, 47.15, -58.94, 55.11  # a box around your area, in degrees
EXPECTED = 963          # how many features you expect (rows after your filters)
POPUP_FIELDS = []  # which fields you want to show in the popup
# ----------------------------------------------------

back = gpd.read_file(FILE)
problems = []
if len(back) != EXPECTED:
    problems.append(f"{len(back)} features, but you expected {EXPECTED}.")
if back.crs is None or back.crs.to_epsg() != 4326:
    problems.append(f"The file is in {back.crs}, not longitude and latitude (EPSG:4326).")
minx, miny, maxx, maxy = back.total_bounds
if not (WEST <= minx and maxx <= EAST and SOUTH <= miny and maxy <= NORTH):
    problems.append(f"Some features are outside your box: x {minx:.3f} to {maxx:.3f}, y {miny:.3f} to {maxy:.3f}.")
missing = [f for f in POPUP_FIELDS if f not in back.columns]
if missing:
    problems.append(f"Missing popup fields: {missing}.")
mb = os.path.getsize(FILE) / 1_000_000
if mb > 5:
    problems.append(f"The file is {mb:.1f} MB; over about 5 MB the page gets slow. Filter or simplify.")
print("Geometry types:", sorted(back.geom_type.unique()))
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

Geometry types: ['MultiPolygon', 'Polygon']
All checks passed


In [ ]:
Manicouagan_disturbance = export_disturbance_layer(
    "../Data/Manicouagan2020.gdb",
    "../Data/Take2/",
    tolerance=25
)

In [ ]:
Manouane_disturbance = export_disturbance_layer(
    "../Data/Manouane2020.gdb",
    "../Data/Take2/",
    tolerance=25
)

In [9]:
# ---- settings: change these for your own data ----
FILE = "../Data/Take2/Manouane2020_Disturb_Perturb_Poly.geojson"
WEST, EAST, SOUTH, NORTH =  -82.64, 47.15, -58.94, 55.11  # a box around your area, in degrees
EXPECTED = 1673         # how many features you expect (rows after your filters)
POPUP_FIELDS = []  # which fields you want to show in the popup
# ----------------------------------------------------

back = gpd.read_file(FILE)
problems = []
if len(back) != EXPECTED:
    problems.append(f"{len(back)} features, but you expected {EXPECTED}.")
if back.crs is None or back.crs.to_epsg() != 4326:
    problems.append(f"The file is in {back.crs}, not longitude and latitude (EPSG:4326).")
minx, miny, maxx, maxy = back.total_bounds
if not (WEST <= minx and maxx <= EAST and SOUTH <= miny and maxy <= NORTH):
    problems.append(f"Some features are outside your box: x {minx:.3f} to {maxx:.3f}, y {miny:.3f} to {maxy:.3f}.")
missing = [f for f in POPUP_FIELDS if f not in back.columns]
if missing:
    problems.append(f"Missing popup fields: {missing}.")
mb = os.path.getsize(FILE) / 1_000_000
if mb > 5:
    problems.append(f"The file is {mb:.1f} MB; over about 5 MB the page gets slow. Filter or simplify.")
print("Geometry types:", sorted(back.geom_type.unique()))
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

Geometry types: ['MultiPolygon', 'Polygon']
All checks passed


In [ ]:
pipmuacan_disturbance = export_disturbance_layer(
    "../Data/pipmuacan2020.gdb",
    "../Data/Take2/",
    tolerance=25
)

In [ ]:
Quebec_disturbance = export_disturbance_layer(
    "../Data/Quebec2020.gdb",
    "../Data/Take2/",
    tolerance=80
)

The Quebec dataset overlaps with a lot ot the other data sets so this is to remove the overlapping information


In [ ]:
quebec = gpd.read_file(
    "../Data/Take2/Quebec2020_Disturb_Perturb_Poly.geojson"
)

manicouagan = gpd.read_file(
    "../Data/Take2/Manicouagan2020_Disturb_Perturb_Poly.geojson"
)

manouane = gpd.read_file(
    "../Data/Take2/Manouane2020_Disturb_Perturb_Poly.geojson"
)

pipmuacan = gpd.read_file(
    "../Data/Take2/Pipmuacan2020_Disturb_Perturb_Poly.geojson"
)


In [ ]:
quebec_projected = quebec.to_crs("EPSG:32198")

manicouagan_projected = manicouagan.to_crs("EPSG:32198")

manouane_projected = manouane.to_crs("EPSG:32198")

pipmuacan_projected = pipmuacan.to_crs("EPSG:32198")

In [ ]:
regional_geometries = pd.concat(
    [manicouagan_projected, manouane_projected, pipmuacan_projected],
    ignore_index=True
)

regional_union = regional_geometries.geometry.union_all()

Removing the overlapping data points between the different data sets 

In [ ]:
# 1. Identify features that overlap
overlap_mask = quebec_projected.geometry.intersects(regional_union)

# 2. Calculate the total overlap area
overlap_area = quebec_projected.loc[
    overlap_mask,
    "geometry"
].apply(
    lambda geometry: geometry.intersection(regional_union).area
).sum()

quebec_area = quebec_projected.geometry.area.sum()

print("Quebec features touching regional layers:", overlap_mask.sum())
print("Overlap area:", overlap_area)
print("Percentage overlapping:", (overlap_area / quebec_area) * 100)


# 3. Remove overlapping areas from Quebec geometries
quebec_no_overlap = quebec_projected.copy()

quebec_no_overlap["geometry"] = (
    quebec_no_overlap.geometry.difference(regional_union)
)

# 4. Remove any empty geometries
quebec_no_overlap = quebec_no_overlap[
    ~quebec_no_overlap.geometry.is_empty
].copy()

print("Original Quebec features:", len(quebec_projected))
print("Remaining Quebec features:", len(quebec_no_overlap))

Then creating two new geojson of the quebec data without the overlapping information and seperating Cutblocks and the rest of the distubance classes

In [ ]:
print("Original CRS:", quebec_no_overlap.crs),
print("Original features:", len(quebec_no_overlap)),

# Find all features whose Class contains the word cut\n",
cutblock_mask = (
    quebec_no_overlap["Class"].astype(str).str.contains("Cut", case=False, na=False)
)
    
print("Cut-block features:", cutblock_mask.sum())

# 2. Split the merged data into two GeoDataFrames\n",
cutblocks = quebec_no_overlap[cutblock_mask].copy()
without_cutblocks = quebec_no_overlap[~cutblock_mask].copy()\

print("Cut-block features:", len(cutblocks))
print("Other features:", len(without_cutblocks))

# 3. Keep only the fields needed by the map\n",
columns_to_keep = ["Class","source_dataset", "geometry"]

# 4. Convert to metres before simplifying\n",
cutblocks = cutblocks.to_crs("EPSG:32198")
without_cutblocks = without_cutblocks.to_crs("EPSG:32198")

# 5. Simplify both geometries by 10 metres,
cutblocks["geometry"] = cutblocks.geometry.simplify(10, preserve_topology=True)
without_cutblocks["geometry"] = without_cutblocks.geometry.simplify(10, preserve_topology=True)

  # 6. Convert both final files to EPSG:4326\n",
cutblocks = cutblocks.to_crs("EPSG:4326")
without_cutblocks = without_cutblocks.to_crs("EPSG:4326")

# 7. Save two separate GeoJSON files\n",
cutblocks.to_file("../Data/Take2/Quebec_HU_Cutblocks.geojson",
    driver="GeoJSON",
    COORDINATE_PRECISION=6,
)
without_cutblocks.to_file("../Data/Take2/Quebec_Without_Cutblocks.geojson",
    driver="GeoJSON",
    COORDINATE_PRECISION=6,
)

       
print("Cleaned features:", len(cutblocks) + len(without_cutblocks)),
print("Cleaned columns:", cutblocks.columns.tolist()), without_cutblocks.columns.tolist()

In [ ]:
ValdOr_disturbance = export_disturbance_layer(
    "../Data/ValdOr2020.gdb",
    "../Data/Take2/",
    tolerance=25
)

Creating combined data sets to show some of the different layers. An identifer for each layer to make sure they dont loose track of the original location and population. As i am going to merge them by layer so I can have them overlay over the map.

Layer: population Range == combined_Range 
Layer : Human disturbances == combined_Hu_Dis
Layer : Linear disturbances (roads, pipelines ect.) == combined_Lin_Dis
Layer: Human disturbances with a 500 m buffer included == combined_Dis_Buff
Layer ; Human disturbances with a 500m buffer plus fires == combined_Dis_Buff_fire 



combined_Range = []
for gdb_path in gdb_paths:
    layer_info = gpd.list_layers(str(gdb_path))
    matching_layers = [
        name for name in layer_info["name"]
        if name.endswith("_Range_Aire")
    ]
   
    print(gdb_path.name, matching_layers)
    if len(matching_layers) != 1:
        print("Check this file")
        continue

    layer_name = matching_layers[0]
    layer_data = gpd.read_file(
           str(gdb_path),
        layer=layer_name
    )

    layer_data["source_dataset"] = gdb_path.stem
    layer_data["source_layer"] = layer_name
    layer_data["source_feature_id"] = layer_data.index

    combined_Range.append(layer_data)

combined = gpd.GeoDataFrame(
    pd.concat(combined_Range, ignore_index=True),
    crs=combined_Range[0].crs
)

print("Combined features:", len(combined))
print(combined["source_dataset"].value_counts())

In [ ]:

combined_Hu_Dis = []
for gdb_path in gdb_paths:
    layer_info = gpd.list_layers(str(gdb_path))
    matching_layers = [
        name for name in layer_info["name"]
        if name.endswith("_Disturb_Perturb_Poly")
    ]
   
    print(gdb_path.name, matching_layers)
    if len(matching_layers) != 1:
        print("Check this file")
        continue

    layer_name = matching_layers[0]
    layer_data = gpd.read_file(
           str(gdb_path),
        layer=layer_name
    )

    layer_data["source_dataset"] = gdb_path.stem
    layer_data["source_layer"] = layer_name
    layer_data["source_feature_id"] = layer_data.index

    combined_Hu_Dis.append(layer_data)

combined = gpd.GeoDataFrame(
    pd.concat(combined_Hu_Dis, ignore_index=True),
    crs=combined_Hu_Dis[0].crs
)

print("Combined features:", len(combined))
print(combined["source_dataset"].value_counts())

In [ ]:

combined_Lin_Dis = []
for gdb_path in gdb_paths:
    layer_info = gpd.list_layers(str(gdb_path))
    matching_layers = [
        name for name in layer_info["name"]
        if name.endswith("_Disturb_Perturb_Line")
    ]
   
    print(gdb_path.name, matching_layers)
    if len(matching_layers) != 1:
        print("Check this file")
        continue

    layer_name = matching_layers[0]
    layer_data = gpd.read_file(
           str(gdb_path),
        layer=layer_name
    )

    layer_data["source_dataset"] = gdb_path.stem
    layer_data["source_layer"] = layer_name
    layer_data["source_feature_id"] = layer_data.index

    combined_Lin_Dis.append(layer_data)

combined = gpd.GeoDataFrame(
    pd.concat(combined_Lin_Dis, ignore_index=True),
    crs=combined_Lin_Dis[0].crs
)

print("Combined features:", len(combined))
print(combined["source_dataset"].value_counts())

In [ ]:

combined_Dis_Buff = []
for gdb_path in gdb_paths:
    layer_info = gpd.list_layers(str(gdb_path))
    matching_layers = [
        name for name in layer_info["name"]
        if name.endswith("_Anthro_500m")
    ]
   
    print(gdb_path.name, matching_layers)
    if len(matching_layers) != 1:
        print("Check this file")
        continue

    layer_name = matching_layers[0]
    layer_data = gpd.read_file(
           str(gdb_path),
        layer=layer_name
    )

    layer_data["source_dataset"] = gdb_path.stem
    layer_data["source_layer"] = layer_name
    layer_data["source_feature_id"] = layer_data.index

    combined_Dis_Buff.append(layer_data)

combined = gpd.GeoDataFrame(
    pd.concat(combined_Dis_Buff, ignore_index=True),
    crs=combined_Dis_Buff[0].crs
)

print("Combined features:", len(combined))
print(combined["source_dataset"].value_counts())

In [ ]:

combined_Dis_Buff_fire = []
for gdb_path in gdb_paths:
    layer_info = gpd.list_layers(str(gdb_path))
    matching_layers = [
        name for name in layer_info["name"]
        if name.endswith("_Anthro_500m_AndFire_EtFeu")
    ]
   
    print(gdb_path.name, matching_layers)
    if len(matching_layers) != 1:
        print("Check this file")
        continue

    layer_name = matching_layers[0]
    layer_data = gpd.read_file(
           str(gdb_path),
        layer=layer_name
    )

    layer_data["source_dataset"] = gdb_path.stem
    layer_data["source_layer"] = layer_name
    layer_data["source_feature_id"] = layer_data.index

    combined_Dis_Buff_fire.append(layer_data)

combined = gpd.GeoDataFrame(
    pd.concat(combined_Dis_Buff_fire, ignore_index=True),
    crs=combined_Dis_Buff_fire[0].crs
)

print("Combined features:", len(combined))
print(combined["source_dataset"].value_counts())

Creating the Geojsons for each layer I would like to use in my final project

In [ ]:
combined_range = gpd.GeoDataFrame(
      pd.concat(combined_Range, ignore_index=True),
      crs=combined_Range[0].crs
)
combined_range["geometry"] = combined_range.geometry.simplify(5)
combined_range = combined_range.to_crs("EPSG:4326") 
combined_range.to_file("../Data/Caribou_Range_Combined.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

This creates the combined human disturbances data set

In [ ]:
combined_hu_dis = gpd.GeoDataFrame(
      pd.concat(combined_Hu_Dis, ignore_index=True),
      crs=combined_Hu_Dis[0].crs
)
combined_hu_dis["geometry"] = combined_hu_dis.geometry.simplify( tolerance=25,
        preserve_topology=True)
combined_hu_dis = combined_hu_dis.to_crs("EPSG:4326") 
combined_hu_dis.to_file("../Data/Caribou_Hu_Dis_Combined.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

Simplifying and filtering cause she is a big gall! 

In [ ]:
print("Original CRS:", combined_hu_dis.crs),
print("Original features:", len(combined_hu_dis)),

# Find all features whose Class contains the word cut\n",
cutblock_mask = (
    combined_hu_dis["Class"].astype(str).str.contains("Cut", case=False, na=False)
)
    
print("Cut-block features:", cutblock_mask.sum())

# 2. Split the merged data into two GeoDataFrames\n",
cutblocks = combined_hu_dis[cutblock_mask].copy()
without_cutblocks = combined_hu_dis[~cutblock_mask].copy()\

print("Cut-block features:", len(cutblocks))
print("Other features:", len(without_cutblocks))

# 3. Keep only the fields needed by the map\n",
columns_to_keep = ["Class","source_dataset", "geometry"]

# 4. Convert to metres before simplifying\n",
cutblocks = cutblocks.to_crs("EPSG:32198")
without_cutblocks = without_cutblocks.to_crs("EPSG:32198")

# 5. Simplify both geometries by 10 metres,
cutblocks["geometry"] = cutblocks.geometry.simplify(10, preserve_topology=True)
without_cutblocks["geometry"] = without_cutblocks.geometry.simplify(10, preserve_topology=True)

  # 6. Convert both final files to EPSG:4326\n",
cutblocks = cutblocks.to_crs("EPSG:4326")
without_cutblocks = without_cutblocks.to_crs("EPSG:4326")

# 7. Save two separate GeoJSON files\n",
cutblocks.to_file("../Data/Cleaned/Combined_HU_Cutblocks.geojson",
    driver="GeoJSON",
    COORDINATE_PRECISION=6,
)
without_cutblocks.to_file("../Data/Cleaned/Quebec_Without_Cutblocks.geojson",
    driver="GeoJSON",
    COORDINATE_PRECISION=6,
)

       
print("Cleaned features:", len(cutblocks) + len(without_cutblocks)),
print("Cleaned columns:", cutblocks.columns.tolist()), without_cutblocks.columns.tolist()

In [ ]:
# ---- settings: change these for your own data ----
FILE = "../Data/Cleaned/Caribou_Hu_Dis_Combined_Simplified.geojson"
WEST, EAST, SOUTH, NORTH =  -82.64, 47.329, -58.94, 55.4  # a box around your area, in degrees
EXPECTED = 30511         # how many features you expect (rows after your filters)
POPUP_FIELDS = ["Class"]  # which fields you want to show in the popup
# ----------------------------------------------------

back = gpd.read_file(FILE)
problems = []
if len(back) != EXPECTED:
    problems.append(f"{len(back)} features, but you expected {EXPECTED}.")
if back.crs is None or back.crs.to_epsg() != 4326:
    problems.append(f"The file is in {back.crs}, not longitude and latitude (EPSG:4326).")
minx, miny, maxx, maxy = back.total_bounds
if not (WEST <= minx and maxx <= EAST and SOUTH <= miny and maxy <= NORTH):
    problems.append(f"Some features are outside your box: x {minx:.3f} to {maxx:.3f}, y {miny:.3f} to {maxy:.3f}.")
missing = [f for f in POPUP_FIELDS if f not in back.columns]
if missing:
    problems.append(f"Missing popup fields: {missing}.")
mb = os.path.getsize(FILE) / 1_000_000
if mb > 5:
    problems.append(f"The file is {mb:.1f} MB; over about 5 MB the page gets slow. Filter or simplify.")
print("Geometry types:", sorted(back.geom_type.unique()))
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

This created the layer for the Linear disturbances. 

In [ ]:
combined_LIN = gpd.GeoDataFrame(
      pd.concat(combined_Lin_Dis, ignore_index=True),
      crs=combined_Lin_Dis[0].crs
)
combined_LIN["geometry"] = combined_LIN.geometry.simplify(5)
combined_LIN = combined_LIN.to_crs("EPSG:4326") 
combined_LIN.to_file("../Data/Caribou_Lin_Dis_Combined.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

In [ ]:
# ---- settings: change these for your own data ----
FILE = "../Data/Cleaned/Caribou_Lin_Dis_Combined.geojson"
WEST, EAST, SOUTH, NORTH =  -82.64, 47.329, -58.94, 55.7  # a box around your area, in degrees
EXPECTED = 41677         # how many features you expect (rows after your filters)
POPUP_FIELDS = ["Class", "Year_added"]  # which fields you want to show in the popup
# ----------------------------------------------------

back = gpd.read_file(FILE)
problems = []
if len(back) != EXPECTED:
    problems.append(f"{len(back)} features, but you expected {EXPECTED}.")
if back.crs is None or back.crs.to_epsg() != 4326:
    problems.append(f"The file is in {back.crs}, not longitude and latitude (EPSG:4326).")
minx, miny, maxx, maxy = back.total_bounds
if not (WEST <= minx and maxx <= EAST and SOUTH <= miny and maxy <= NORTH):
    problems.append(f"Some features are outside your box: x {minx:.3f} to {maxx:.3f}, y {miny:.3f} to {maxy:.3f}.")
missing = [f for f in POPUP_FIELDS if f not in back.columns]
if missing:
    problems.append(f"Missing popup fields: {missing}.")
mb = os.path.getsize(FILE) / 1_000_000
if mb > 5:
    problems.append(f"The file is {mb:.1f} MB; over about 5 MB the page gets slow. Filter or simplify.")
print("Geometry types:", sorted(back.geom_type.unique()))
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

The combined layer for the disturbances map with 500 m buffer.

In [ ]:
combined_dis_buff = gpd.GeoDataFrame(
      pd.concat(combined_Dis_Buff, ignore_index=True),
      crs=combined_Dis_Buff[0].crs
)
combined_dis_buff["geometry"] = combined_dis_buff.geometry.simplify(5)
combined_dis_buff = combined_dis_buff.to_crs("EPSG:4326") 
combined_dis_buff.to_file("../Data/Caribou_Dis_Buff_Combined.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

In [ ]:
# ---- settings: change these for your own data ----
FILE = "../Data/Cleaned/Caribou_Dis_Buff_Combined.geojson"
WEST, EAST, SOUTH, NORTH =  -82.64, 47.329, -58.94, 55.7  # a box around your area, in degrees
EXPECTED = 6       # how many features you expect (rows after your filters)
POPUP_FIELDS = []  # which fields you want to show in the popup
# ----------------------------------------------------

back = gpd.read_file(FILE)
problems = []
if len(back) != EXPECTED:
    problems.append(f"{len(back)} features, but you expected {EXPECTED}.")
if back.crs is None or back.crs.to_epsg() != 4326:
    problems.append(f"The file is in {back.crs}, not longitude and latitude (EPSG:4326).")
minx, miny, maxx, maxy = back.total_bounds
if not (WEST <= minx and maxx <= EAST and SOUTH <= miny and maxy <= NORTH):
    problems.append(f"Some features are outside your box: x {minx:.3f} to {maxx:.3f}, y {miny:.3f} to {maxy:.3f}.")
missing = [f for f in POPUP_FIELDS if f not in back.columns]
if missing:
    problems.append(f"Missing popup fields: {missing}.")
mb = os.path.getsize(FILE) / 1_000_000
if mb > 5:
    problems.append(f"The file is {mb:.1f} MB; over about 5 MB the page gets slow. Filter or simplify.")
print("Geometry types:", sorted(back.geom_type.unique()))
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

The combined layer for anthropogenic disturbances with buffer as well as fire disturbances 

In [ ]:
combined_dis_buff_fire = gpd.GeoDataFrame(
      pd.concat(combined_Dis_Buff_fire, ignore_index=True),
      crs=combined_Dis_Buff_fire[0].crs
)
combined_dis_buff_fire["geometry"] = combined_dis_buff_fire.geometry.simplify(5)
combined_dis_buff_fire = combined_dis_buff_fire.to_crs("EPSG:4326") 
combined_dis_buff_fire.to_file("../Data/Cleaned/Caribou_Dis_Buff_Fire_Combined.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

In [ ]:
# ---- settings: change these for your own data ----
FILE = "../Data/Cleaned/Caribou_Dis_Buff_Fire_Combined.geojson"
WEST, EAST, SOUTH, NORTH =  -82.64, 47.329, -58.94, 55.7  # a box around your area, in degrees
EXPECTED = 6       # how many features you expect (rows after your filters)
POPUP_FIELDS = []  # which fields you want to show in the popup
# ----------------------------------------------------

back = gpd.read_file(FILE)
problems = []
if len(back) != EXPECTED:
    problems.append(f"{len(back)} features, but you expected {EXPECTED}.")
if back.crs is None or back.crs.to_epsg() != 4326:
    problems.append(f"The file is in {back.crs}, not longitude and latitude (EPSG:4326).")
minx, miny, maxx, maxy = back.total_bounds
if not (WEST <= minx and maxx <= EAST and SOUTH <= miny and maxy <= NORTH):
    problems.append(f"Some features are outside your box: x {minx:.3f} to {maxx:.3f}, y {miny:.3f} to {maxy:.3f}.")
missing = [f for f in POPUP_FIELDS if f not in back.columns]
if missing:
    problems.append(f"Missing popup fields: {missing}.")
mb = os.path.getsize(FILE) / 1_000_000
if mb > 5:
    problems.append(f"The file is {mb:.1f} MB; over about 5 MB the page gets slow. Filter or simplify.")
print("Geometry types:", sorted(back.geom_type.unique()))
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

import geopandas as gpd

gdb_path = "../Data/Charlevoix2020.gdb"

gpd.list_layers(gdb_path)

Now to add the Buffer layer to the map and simplify it so it only shows the area that is not on the map already 

In [4]:
Buffer = gpd.read_file(
    "../Data/Cleaned/Caribou_Dis_Buff_Combined.geojson"
).to_crs("EPSG:32198") 
QC_projected_cutblocks = gpd.read_file(
    "../Data/Take2/Quebec_HU_Cutblocks.geojson"
).to_crs("EPSG:32198")
QC_projected_without = gpd.read_file(
    "../Data/Take2/Quebec_Without_Cutblocks.geojson"
).to_crs("EPSG:32198")

DataSourceError: '../Data/Cleaned/Caribou_Dis_Buff_Combined.geojson' not recognized as being in a supported file format.; It might help to specify the correct driver explicitly by prefixing the file path with '<DRIVER>:', e.g. 'CSV:path'.

In [ ]:
regional_geometries = pd.concat(
    [manicouagan_projected, manouane_projected, pipmuacan_projected, QC_projected_cutblocks, QC_projected_without],
    ignore_index=True
)

regional_union2 = regional_geometries.geometry.union_all()

In [ ]:
# 2. Remove overlapping areas using the regional union
buffer_no_overlap = Buffer.copy()

buffer_no_overlap["geometry"] = (
    Buffer.geometry.difference(regional_union2)
)


# 3. Remove empty geometries
buffer_no_overlap = buffer_no_overlap[
    ~buffer_no_overlap.geometry.is_empty
].copy()


# 4. Check feature counts
print("Original buffer features:", len(Buffer))
print("Remaining buffer features:", len(buffer_no_overlap))


# 5. Convert back to geographic coordinates
buffer_no_overlap = buffer_no_overlap.to_crs("EPSG:4326")


# 6. Export as GeoJSON
buffer_no_overlap.to_file(
    "../Data/Take2/Buffer_Without_Regional_Overlap.geojson",
    driver="GeoJSON",
    COORDINATE_PRECISION=5
)

Add the fire disturbance data 

In [5]:
os.environ["OGR_GEOJSON_MAX_OBJ_SIZE"] = "0" 
#file was too large to open used this to open it so I can shrink it down

In [ ]:
Fire = gpd.read_file(
    "../Data/Cleaned/Caribou_Dis_Buff_Fire_Combined.geojson"
).to_crs("EPSG:32198")
Buffer = gpd.read_file(
    "../Data/Cleaned/Caribou_Dis_Buff_Combined.geojson"
).to_crs("EPSG:32198") 

In [10]:
Fire.geometry = Fire.geometry.make_valid()
Fire = Fire[~Fire.geometry.is_empty]

In [11]:
print("Invalid geometries:", (~Fire.geometry.is_valid).sum())
print("Empty geometries:", Fire.geometry.is_empty.sum())
print("Geometry types:", Fire.geometry.geom_type.value_counts())

Invalid geometries: 0
Empty geometries: 0
Geometry types: MultiPolygon          5
GeometryCollection    1
Name: count, dtype: int64


In [12]:
Buffer.geometry = Buffer.geometry.make_valid()

# Remove empty geometries
Buffer = Buffer[~Buffer.geometry.is_empty]

print("Invalid geometries:", (~Buffer.geometry.is_valid).sum())
print("Empty geometries:", Buffer.geometry.is_empty.sum())
print("Geometry types:", Buffer.geometry.geom_type.value_counts())

buffer_union = Buffer.geometry.union_all()

print("Union valid:", buffer_union.is_valid)

Invalid geometries: 0
Empty geometries: 0
Geometry types: MultiPolygon    6
Name: count, dtype: int64
Union valid: True


In [1]:
# 2. Remove overlapping areas using the regional union
Fire_no_overlap = Fire.copy()

Fire_no_overlap["geometry"] = (
    Fire.geometry.difference(buffer_union))


# 3. Remove empty geometries
Fire_no_overlap = Fire_no_overlap[
    ~Fire_no_overlap.geometry.is_empty
].copy()


# 4. Check feature counts
print("Original Fire features:", len(Fire))
print("Remaining Fire features:", len(Fire_no_overlap))
print("Original area coverage", Fire.geometry.area.sum())
print("Remaining area coverage", Fire_no_overlap.geometry.area.sum())


# 5. Convert back to geographic coordinates
Fire_no_overlap = Fire_no_overlap.to_crs("EPSG:4326")


# 6. Export as GeoJSON
Fire_no_overlap.to_file(
    "../Data/Take2/Fire_Without_Overlap.geojson",
    driver="GeoJSON",
    COORDINATE_PRECISION=5
)

NameError: name 'Fire' is not defined

In [15]:
# Create a simplified copy
Fire_simplified = Fire.copy().to_crs("EPSG:32198")

Fire_simplified.geometry = Fire_simplified.geometry.simplify(
    tolerance=100,
    preserve_topology=True
)

# Remove unnecessary columns (keep geometry only for now)
Fire_simplified = Fire_simplified[["geometry"]]

# Save as compact GeoJSON
Fire_simplified.to_file(
    "../Data/Cleaned/Caribou_Fire_Web.geojson",
    driver="GeoJSON",
    index=False
)